# Data Preprocessing

This notebook prepares the dataset for machine learning by performing:
- feature selection
- categorical encoding
- train-test splitting
- feature scaling
- preprocessing for model training

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/ai4i2020.csv")

df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 1. Initial Data Check

Before preprocessing, we first inspect the dataset structure again.  
This helps us confirm the number of rows and columns, check data types, detect missing values or duplicates, and understand the class distribution of the target variable.

In [2]:
# Basic dataset information
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nTarget distribution:")
print(df["Machine failure"].value_counts())

print("\nTarget distribution in percentage:")
print(df["Machine failure"].value_counts(normalize=True) * 100)

Dataset shape: (10000, 14)

Column names:
['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

Data types:
UDI                          int64
Product ID                     str
Type                           str
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object

Missing values:
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            

## 2. Feature and Target Selection

In this step, we separate the dataset into input features `X` and target variable `y`.

The target variable is `Machine failure`, because the goal of this project is to predict whether a machine will fail or not.

Some columns are removed from the input features. `UDI` and `Product ID` are identifiers and do not provide useful machine behavior information. The columns `TWF`, `HDF`, `PWF`, `OSF`, and `RNF` describe specific failure types, so they are not used as input features because they could leak information about the target.

In [3]:
# Define target column
target_col = "Machine failure"

# Columns to remove from the input features
identifier_cols = ["UDI", "Product ID"]
leakage_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]

cols_to_drop = identifier_cols + leakage_cols + [target_col]

# Create feature matrix X and target vector y
X = df.drop(columns=cols_to_drop)
y = df[target_col]

print("Feature columns:")
print(X.columns.tolist())

print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget distribution in percentage:")
print(y.value_counts(normalize=True) * 100)

X.head()

Feature columns:
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

Feature shape: (10000, 6)
Target shape: (10000,)

Target distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64

Target distribution in percentage:
Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


## 3. Train-Test Split

The dataset is now split into training and test sets.

A stratified split is used because the target variable `Machine failure` is highly imbalanced. This keeps the percentage of failed and non-failed machines similar in both the training and test sets.

The test set is kept separate and will only be used later for final model evaluation.

In [4]:
from sklearn.model_selection import train_test_split

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True) * 100)

print("\nTest target distribution:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True) * 100)

X_train shape: (8000, 6)
X_test shape: (2000, 6)
y_train shape: (8000,)
y_test shape: (2000,)

Training target distribution:
Machine failure
0    7729
1     271
Name: count, dtype: int64
Machine failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64

Test target distribution:
Machine failure
0    1932
1      68
Name: count, dtype: int64
Machine failure
0    96.6
1     3.4
Name: proportion, dtype: float64


## 4. Categorical Encoding and Feature Scaling

The dataset contains one categorical feature, `Type`, and several numerical features.

Machine learning models cannot directly use text categories such as `L`, `M`, and `H`, so the `Type` column is converted using one-hot encoding.

The numerical features are scaled using standardization. This gives the numerical columns a similar scale, which is especially important for models such as logistic regression, SVM, KNN, and neural networks.

The preprocessing transformer is fitted only on the training data. The same fitted transformer is then applied to the test data to avoid data leakage.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify categorical and numerical features
categorical_features = ["Type"]
numeric_features = X_train.drop(columns=categorical_features).columns.tolist()

print("Categorical features:", categorical_features)
print("Numerical features:", numeric_features)

# Version-compatible OneHotEncoder
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

# Create preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", one_hot_encoder, categorical_features)
    ]
)

# Fit only on training data, then transform both train and test data
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Get final feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

# Convert back to DataFrames for easier inspection
X_train_preprocessed = pd.DataFrame(
    X_train_preprocessed,
    columns=feature_names,
    index=X_train.index
)

X_test_preprocessed = pd.DataFrame(
    X_test_preprocessed,
    columns=feature_names,
    index=X_test.index
)

print("Preprocessed training shape:", X_train_preprocessed.shape)
print("Preprocessed test shape:", X_test_preprocessed.shape)

X_train_preprocessed.head()

Categorical features: ['Type']
Numerical features: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
Preprocessed training shape: (8000, 8)
Preprocessed test shape: (2000, 8)


,num__Air temperature [K],num__Process temperature [K],num__Rotational speed [rpm],num__Torque [Nm],num__Tool wear [min],cat__Type_H,cat__Type_L,cat__Type_M
4058,0.998914,0.604282,-0.460607,0.718305,-0.843997,0.0,0.0,1.0
1221,-1.505194,-1.153260,-0.775574,0.638456,0.382263,0.0,0.0,1.0
6895,0.498092,1.077466,-1.007654,0.558607,0.460870,0.0,0.0,1.0
9863,-0.553633,-0.139294,-0.709265,1.626586,-0.372359,0.0,1.0,0.0
8711,-1.455112,-1.018064,1.070019,-1.128202,-0.906882,0.0,1.0,0.0


## 5. Save Preprocessed Train and Test Sets

The preprocessing step is now complete. The training and test features have been encoded and scaled, while the target values remain unchanged.

The processed datasets are saved into the `data/processed` folder so they can be used directly in the model training notebook.

The fitted preprocessing transformer is also saved. This is important because the same preprocessing steps must be applied later when making predictions on new data.

In [6]:
from pathlib import Path
import joblib

# Save into the existing processed data folder
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Combine preprocessed features with target
train_processed = X_train_preprocessed.copy()
train_processed["Machine failure"] = y_train.values

test_processed = X_test_preprocessed.copy()
test_processed["Machine failure"] = y_test.values

# Save processed datasets
train_processed.to_csv(processed_dir / "train_processed.csv", index=False)
test_processed.to_csv(processed_dir / "test_processed.csv", index=False)

# Save the fitted preprocessor in the same processed folder
joblib.dump(preprocessor, processed_dir / "preprocessor.joblib")

print("Saved files:")
print((processed_dir / "train_processed.csv").resolve())
print((processed_dir / "test_processed.csv").resolve())
print((processed_dir / "preprocessor.joblib").resolve())

print("\nTrain processed shape:", train_processed.shape)
print("Test processed shape:", test_processed.shape)

Saved files:
/Users/aliakbar/Predictive-Maintenance-AI/predictive-maintenance-ai/data/processed/train_processed.csv
/Users/aliakbar/Predictive-Maintenance-AI/predictive-maintenance-ai/data/processed/test_processed.csv
/Users/aliakbar/Predictive-Maintenance-AI/predictive-maintenance-ai/data/processed/preprocessor.joblib

Train processed shape: (8000, 9)
Test processed shape: (2000, 9)
